In [1]:
# ==============================================================================
# CELL 1: TRACK A - HYBRID FRAUD ENGINE (AUTOENCODER + XGBOOST + SR 11-7 PSI)
# ==============================================================================
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import precision_recall_curve, auc, classification_report, confusion_matrix
import xgboost as xgb
from pyspark.sql import functions as F

# 1. Load Stratified ML Training Set (250K rows)
pdf_train = spark.read.table("gold_ml_fraud_train").toPandas()

feature_cols = [
    "step", "amount", "oldbalanceOrg", "newbalanceOrig",
    "oldbalanceDest", "newbalanceDest", "errorBalanceOrig",
    "errorBalanceDest", "origBalanceDrainRatio",
    "isZeroBalanceOrigAfter", "isZeroBalanceDestAfter",
    "isHighRiskChannel", "hourOfDay", "orig_tx_count_1h",
    "orig_tx_sum_1h", "dest_inflow_count_1h", "dest_inflow_sum_1h"
]

# One-hot encode transaction type
pdf_encoded = pd.get_dummies(pdf_train[feature_cols + ["type"]], columns=["type"], drop_first=False)
model_features = [c for c in pdf_encoded.columns]

X = pdf_encoded[model_features].fillna(0).values
y = pdf_train["isFraud"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# 2. Scale features using RobustScaler (resistant to extreme outlier amounts)
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. STAGE 1: Unsupervised Deep Autoencoder (Trained strictly on Legitimate Tx)
X_train_legit = X_train_scaled[y_train == 0]
autoencoder = MLPRegressor(
    hidden_layer_sizes=(16, 6, 16),
    activation="relu",
    max_iter=15,
    batch_size=1024,
    random_state=42
)
autoencoder.fit(X_train_legit, X_train_legit)

# Compute Mean Squared Reconstruction Error as an Anomaly Score
train_recon_err = np.mean(np.square(X_train_scaled - autoencoder.predict(X_train_scaled)), axis=1).reshape(-1, 1)
test_recon_err = np.mean(np.square(X_test_scaled - autoencoder.predict(X_test_scaled)), axis=1).reshape(-1, 1)

X_train_hybrid = np.hstack([X_train, train_recon_err])
X_test_hybrid = np.hstack([X_test, test_recon_err])

# 4. STAGE 2: Cost-Sensitive XGBoost Classifier
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
xgb_model = xgb.XGBClassifier(
    n_estimators=180,
    max_depth=6,
    learning_rate=0.08,
    scale_pos_weight=scale_pos,
    subsample=0.85,
    colsample_bytree=0.85,
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train_hybrid, y_train)

# 5. Evaluate PR-AUC & Optimal F2 Threshold
y_probs = xgb_model.predict_proba(X_test_hybrid)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test, y_probs)
pr_auc = auc(recall, precision)

# Maximize F2-score (prioritizing fraud recall while slashing false positives)
f2_scores = (5 * precision * recall) / (4 * precision + recall + 1e-8)
best_idx = np.argmax(f2_scores)
best_thresh = thresholds[best_idx]
y_pred_opt = (y_probs >= best_thresh).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred_opt).ravel()
fpr = fp / (fp + tn) * 100
recall_pct = tp / (tp + fn) * 100

print(f"=== HYBRID FRAUD ENGINE PERFORMANCE ===")
print(f"PR-AUC Score:         {pr_auc:.4f}")
print(f"Optimal Threshold:    {best_thresh:.4f}")
print(f"Fraud Recall:         {recall_pct:.2f}% (Target >= 95%)")
print(f"False Positive Rate:  {fpr:.2f}% (Down from 93.8% legacy baseline!)")

# 6. SR 11-7 Governance: Population Stability Index (PSI) Check
def calculate_psi(expected, actual, buckets=10):
    breakpoints = np.percentile(expected, np.linspace(0, 100, buckets + 1))
    breakpoints[0], breakpoints[-1] = -np.inf, np.inf
    exp_counts = np.histogram(expected, breakpoints)[0] / len(expected)
    act_counts = np.histogram(actual, breakpoints)[0] / len(actual)
    exp_counts = np.where(exp_counts == 0, 1e-6, exp_counts)
    act_counts = np.where(act_counts == 0, 1e-6, act_counts)
    return np.sum((act_counts - exp_counts) * np.log(act_counts / exp_counts))

psi_val = calculate_psi(xgb_model.predict_proba(X_train_hybrid)[:, 1], y_probs)
print(f"SR 11-7 Score PSI:    {psi_val:.5f} ({'STABLE < 0.10' if psi_val < 0.10 else 'DRIFT ALERT'})")

# 7. Score the 100K Power BI Surveillance Table & Save to Gold
pdf_bi = spark.read.table("gold_fact_fraud_surveillance").toPandas()
pdf_bi_enc = pd.get_dummies(pdf_bi[feature_cols + ["type"]], columns=["type"], drop_first=False)
for col in model_features:
    if col not in pdf_bi_enc.columns:
        pdf_bi_enc[col] = 0
X_bi = pdf_bi_enc[model_features].fillna(0).values
X_bi_scaled = scaler.transform(X_bi)
bi_recon_err = np.mean(np.square(X_bi_scaled - autoencoder.predict(X_bi_scaled)), axis=1).reshape(-1, 1)
X_bi_hybrid = np.hstack([X_bi, bi_recon_err])

pdf_bi["autoencoder_anomaly_score"] = np.round(bi_recon_err.flatten(), 4)
pdf_bi["ml_fraud_probability"] = np.round(xgb_model.predict_proba(X_bi_hybrid)[:, 1], 4)
pdf_bi["ml_fraud_alert"] = (pdf_bi["ml_fraud_probability"] >= best_thresh).astype(int)
pdf_bi["risk_tier"] = np.where(pdf_bi["ml_fraud_probability"] >= 0.85, "P1 - Critical",
                      np.where(pdf_bi["ml_fraud_probability"] >= best_thresh, "P2 - High", "Low"))

spark.createDataFrame(pdf_bi).write.format("delta").mode("overwrite").saveAsTable("gold_fact_fraud_scored")
print("Saved scored surveillance table: gold_fact_fraud_scored (100,000 rows)")


StatementMeta(, 51c3a884-293d-4249-9072-e9dac0bf0223, 3, Finished, Available, Finished, False)

2026-11-12 08:00:41 [INFO] setting mlflow tracking uri to sds://api.fabric.microsoft.com/v1/workspaces/6dfa170e-c9c3-45aa-84e3-9a474474c39c/mlflow, change this environment variable if you want log to other places
🏃 View run tidy_stem_679c21y8 at: https://api.fabric.microsoft.com/v1/workspaces/6dfa170e-c9c3-45aa-84e3-9a474474c39c/mlflow/#/experiments/8a2333a9-6e79-4a00-b74c-cc1488309dca/runs/26365d18-8460-4690-bb58-a25397933759
🧪 View experiment at: https://api.fabric.microsoft.com/v1/workspaces/6dfa170e-c9c3-45aa-84e3-9a474474c39c/mlflow/#/experiments/8a2333a9-6e79-4a00-b74c-cc1488309dca


/home/trusted-service-user/cluster-env/trident_env/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (15) reached and the optimization hasn't converged yet.
  warnings.warn(


🏃 View run dreamy_tent_8y3ck3v3 at: https://api.fabric.microsoft.com/v1/workspaces/6dfa170e-c9c3-45aa-84e3-9a474474c39c/mlflow/#/experiments/8a2333a9-6e79-4a00-b74c-cc1488309dca/runs/77ff091e-cde3-49d2-9b8f-d5fd7ac3121d
🧪 View experiment at: https://api.fabric.microsoft.com/v1/workspaces/6dfa170e-c9c3-45aa-84e3-9a474474c39c/mlflow/#/experiments/8a2333a9-6e79-4a00-b74c-cc1488309dca


=== HYBRID FRAUD ENGINE PERFORMANCE ===
PR-AUC Score:         0.9979
Optimal Threshold:    0.9992
Fraud Recall:         99.66% (Target >= 95%)
False Positive Rate:  0.00% (Down from 93.8% legacy baseline!)
SR 11-7 Score PSI:    0.00037 (STABLE < 0.10)
Saved scored surveillance table: gold_fact_fraud_scored (100,000 rows)


In [2]:
# ==============================================================================
# CELL 2: TRACK B - MACRO REVENUE FORECASTING (LIGHTGBM) & MONTE CARLO VaR
# ==============================================================================
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_absolute_percentage_error

# 1. Load ApexFin Segment Financials
pdf_fin = spark.read.table("gold_fact_corporate_financials").toPandas()
pdf_apex = pdf_fin[pdf_fin["ticker"] == "APEX"].sort_values(["segment", "observation_date"]).copy()

# Create lag features per segment
pdf_apex["rev_lag1"] = pdf_apex.groupby("segment")["net_revenue_millions"].shift(1)
pdf_apex["gpv_lag1"] = pdf_apex.groupby("segment")["gpv_millions"].shift(1)
pdf_apex["quarter_num"] = pdf_apex["fiscal_quarter"].str.replace("Q", "").astype(int)
pdf_apex["segment_code"] = pdf_apex["segment"].astype("category").cat.codes

df_model = pdf_apex.dropna(subset=["rev_lag1"]).copy()

macro_features = [
    "segment_code", "quarter_num", "rev_lag1", "gpv_lag1",
    "FEDFUNDS", "cpi_yoy_pct", "yield_spread_10y2y", "ns_level", "ns_slope", "fed_funds_diff1"
]

X_rev = df_model[macro_features]
y_rev = df_model["net_revenue_millions"]

# 2. Train LightGBM Revenue Regressor
lgb_rev = lgb.LGBMRegressor(
    n_estimators=120,
    learning_rate=0.06,
    max_depth=4,
    num_leaves=12,
    min_child_samples=5,
    random_state=42,
    verbose=-1
)
lgb_rev.fit(X_rev, y_rev)

mape = mean_absolute_percentage_error(y_rev, lgb_rev.predict(X_rev)) * 100
print(f"=== REVENUE FORECASTING ENGINE ===")
print(f"In-Sample Segment Revenue MAPE: {mape:.2f}% (Target <= 4.2%)")

# 3. Generate 4-Quarter Forward Forecast Across 3 Macro Scenarios
latest_q = df_model[df_model["observation_date"] == df_model["observation_date"].max()].copy()
scenarios = [
    ("Baseline Consensus", 0.0, 0.0),
    ("Hawkish Shock (+100 bps Fed Rate)", +1.00, +0.40),
    ("Dovish Easing (-100 bps Fed Rate)", -1.00, -0.30)
]

forecast_rows = []
for scen_name, rate_delta, cpi_delta in scenarios:
    for _, row in latest_q.iterrows():
        curr_rev = row["net_revenue_millions"]
        curr_gpv = row["gpv_millions"]
        q_num = int(row["quarter_num"])
        for h in range(1, 5):
            next_q = ((q_num + h - 1) % 4) + 1
            feat_dict = pd.DataFrame([{
                "segment_code": row["segment_code"],
                "quarter_num": next_q,
                "rev_lag1": curr_rev,
                "gpv_lag1": curr_gpv,
                "FEDFUNDS": row["FEDFUNDS"] + (rate_delta * (h / 4.0)),
                "cpi_yoy_pct": row["cpi_yoy_pct"] + (cpi_delta * (h / 4.0)),
                "yield_spread_10y2y": row["yield_spread_10y2y"] - (rate_delta * 0.4),
                "ns_level": row["ns_level"],
                "ns_slope": row["ns_slope"] + rate_delta,
                "fed_funds_diff1": rate_delta / 4.0
            }])
            pred_rev = float(lgb_rev.predict(feat_dict)[0])
            forecast_rows.append({
                "scenario": scen_name,
                "segment": row["segment"],
                "forecast_horizon": f"+{h}Q",
                "projected_revenue_millions": round(pred_rev, 2),
                "simulated_fed_funds": round(feat_dict["FEDFUNDS"].iloc[0], 2),
                "simulated_cpi_yoy": round(feat_dict["cpi_yoy_pct"].iloc[0], 2)
            })
            curr_rev = pred_rev

pdf_forecast = pd.DataFrame(forecast_rows)
spark.createDataFrame(pdf_forecast).write.format("delta").mode("overwrite").saveAsTable("gold_fact_revenue_forecast")

# 4. 10,000-Path Monte Carlo Simulation for 95% Quarterly Revenue VaR
np.random.seed(42)
baseline_q1_total = pdf_forecast[(pdf_forecast["scenario"] == "Baseline Consensus") & (pdf_forecast["forecast_horizon"] == "+1Q")]["projected_revenue_millions"].sum()
simulated_paths = np.random.normal(loc=baseline_q1_total, scale=baseline_q1_total * 0.048, size=10_000)
var_95 = baseline_q1_total - np.percentile(simulated_paths, 5)
cvar_95 = baseline_q1_total - simulated_paths[simulated_paths <= np.percentile(simulated_paths, 5)].mean()

print(f"Baseline +1Q Total Projected Revenue: ${baseline_q1_total:.2f}M")
print(f"95% Quarterly Revenue VaR Downside:   -${var_95:.2f}M")
print(f"95% Expected Shortfall (CVaR):        -${cvar_95:.2f}M")
print("Saved gold_fact_revenue_forecast to OneLake.")

StatementMeta(, 51c3a884-293d-4249-9072-e9dac0bf0223, 4, Finished, Available, Finished, False)

🏃 View run frank_fowl_sg7kwhch at: https://api.fabric.microsoft.com/v1/workspaces/6dfa170e-c9c3-45aa-84e3-9a474474c39c/mlflow/#/experiments/8a2333a9-6e79-4a00-b74c-cc1488309dca/runs/051d96c8-3c87-4f84-9917-a06cfd1558e8
🧪 View experiment at: https://api.fabric.microsoft.com/v1/workspaces/6dfa170e-c9c3-45aa-84e3-9a474474c39c/mlflow/#/experiments/8a2333a9-6e79-4a00-b74c-cc1488309dca


=== REVENUE FORECASTING ENGINE ===
In-Sample Segment Revenue MAPE: 0.90% (Target <= 4.2%)
Baseline +1Q Total Projected Revenue: $267.27M
95% Quarterly Revenue VaR Downside:   -$21.23M
95% Expected Shortfall (CVaR):        -$26.62M
Saved gold_fact_revenue_forecast to OneLake.
